# Movie Recommendation System:

Importing libraries:


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

Importing the Dataset:


In [3]:
df=pd.read_csv('movies_metadata.csv')
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [4]:
data= df.copy() # Copying the data to another variable to avoid any changes in the original data

In [5]:
df.info() # Checking the information of the dataset to understand the data types and missing values

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

# Data Pre-Processing:

In [6]:
df.isnull().sum() # Checking for missing values in the dataset

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

Some coumns are having huge volume of null values which we have to handel.

In [7]:
df.duplicated().sum() # Checking for duplicate rows in the dataset

np.int64(13)

In [8]:
df.drop_duplicates(inplace=True) # Dropping duplicate rows from the dataset

In [9]:
# Selecting relevant columns for the recommendation system
df = df[['title', 'genres', 'overview', 'vote_average','tagline','popularity']]
df.head(5)

,title,genres,overview,vote_average,tagline,popularity
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",7.7,NaN,21.946943
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,6.9,Roll the dice and unleash the excitement!,17.015539
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...,6.5,Still Yelling. Still Fighting. Still Ready for...,11.7129
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom...",6.1,Friends are the people who let you be yourself...,3.859495
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,5.7,Just When His World Is Back To Normal... He's ...,8.387519


In [10]:
df.isnull().sum() # Checking for missing values in the dataset

title               6
genres              0
overview          954
vote_average        6
tagline         25045
popularity          5
dtype: int64

In [11]:
df= df.dropna(subset=['title'])  # Dropping rows with missing values in the 'title' column as it is essential for the recommendation system.

In [12]:
df['overview'] = df['overview'].fillna('')  # Filling missing values in the 'overview' column with empty strings

In [13]:
df.iloc[0]['genres'] # Checking the genres of the first movie in the dataset

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [14]:
import ast
# Function to extract genre names from the genres column
def extract_genres(genres_str):
    try:
        genres_list = ast.literal_eval(genres_str) # ast.literal_eval safely evaluates the string containing a list of dictionaries
        return [genre['name'] for genre in genres_list]
    except (ValueError, SyntaxError):
        return []

df['genres'] = df['genres'].apply(extract_genres) # Applying the function to extract genre names from the genres column

In [15]:
df['genres'].head(5) # Displaying the first 5 rows of the genres column after extracting genre names

0     [Animation, Comedy, Family]
1    [Adventure, Fantasy, Family]
2               [Romance, Comedy]
3        [Comedy, Drama, Romance]
4                        [Comedy]
Name: genres, dtype: object

In [16]:
df['genres'] = df['genres'].apply(lambda x: ' '.join(x)) # Joining the genre names into a single string for each movie
df['genres'].head(5) # Displaying the first 5 rows of the genres column after joining genre names into a single string

0     Animation Comedy Family
1    Adventure Fantasy Family
2              Romance Comedy
3        Comedy Drama Romance
4                      Comedy
Name: genres, dtype: str

In [17]:
df['tagline'] = df['tagline'].fillna('')  # Filling missing values in the 'tagline' column with empty strings

In [18]:
# Now checking for missing values in the dataset after handling missing values in 'title', 'overview', and 'tagline' columns
df.isnull().sum()

title           0
genres          0
overview        0
vote_average    0
tagline         0
popularity      0
dtype: int64

In [19]:
df['tags']= df['overview'] + ' ' + df['genres'] + ' ' + df['tagline'] 
# Creating a new column 'tags' by combining 'overview', 'genres', and 'tagline' columns

In [20]:
df.head(2) # Displaying the first 2 rows of the dataset after handling missing values and extracting genres

,title,genres,overview,vote_average,tagline,popularity,tags
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",7.7,,21.946943,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...,6.9,Roll the dice and unleash the excitement!,17.015539,When siblings Judy and Peter discover an encha...


# Natural Language Processing:

Steps we are going to perform:

-- 1. Removing stopwords

-- 2. Removing punctuations

-- 3. Lammatization

-- 4. Splitting of words

-- 5. Vectorization.

In [21]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
import re

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [22]:
stop_words= set(stopwords.words('english')) # Creating a set of English stopwords for text preprocessing
lemmatizer = WordNetLemmatizer() # Initializing the WordNetLemmatizer for lemmatization of words

In [23]:
def preprocess_text(text):
    # Convert to Lowercase
    text= str(text).lower()

    # Remove Punctuation and Special Characters
    text= re.sub(r'[^a-zA-Z\s]',"",text)

    # Tokenization
    tokens= text.split()

    # Remove Stopwords and Lemmatization
    tokens= [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens) # Joining the tokens back into a single string


In [24]:
df['tags'] = df['tags'].apply(preprocess_text) # Applying the text preprocessing function to the 'tags' column

Now Compare Before and after text preprocessing:


Before

In [25]:
df['tags'][0] # Displaying the 'tags' column of the first movie in the dataset to check the combined text

'led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference animation comedy family'

After

In [26]:
print(df.iloc[0]['tags']) # Displaying the preprocessed 'tags' column of the first movie in the dataset to check the result of text preprocessing

led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference animation comedy family


In [27]:
df= df.reset_index(drop=True) # Resetting the index of the DataFrame after dropping rows with missing values

In [28]:
indices= pd.Series(df.index, index= df['title']).drop_duplicates() # Creating a Series with movie titles as index and their corresponding indices as values, dropping duplicates
indices.head(5) # Displaying the first 5 entries of the indices Series to check the mapping of movie titles to their indices

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
dtype: int64

Vectorization:

It's the process of converting text into numbers.



Types:

1. oneHotEncoding: [[000100],[0010000]] # Variable length size.

2. BagOfWords: select vocabulary size of n, then for each word one vector of n size is created. # Problem is sparse matrix.

3. TFIDF Vectorization: solves the problem of BOW.

ngram sence assuming 2 words as 1: how, are, how are like this.

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf= TfidfVectorizer(max_features= 50000,stop_words='english',ngram_range=(1,2)) # Initializing the TfidfVectorizer with a maximum of 5000 features

In [30]:
tfidf_matrix= tfidf.fit_transform(df['tags']) # Fitting and transforming the 'tags' column to create the TF-IDF matrix (2-dimensional array representation of the text data)
# The shape of the TF-IDF matrix is (number of movies, number of features), (45447, 50000) in this case, where each row represents a movie and each column represents a unique word or n-gram in the 'tags' column.
# where each row represents a movie and each column represents a unique word or n-gram in the 'tags' column.

In [31]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1548114 stored elements and shape (45447, 50000)>

# Creating Model and Cosine Similarity:

Cosine Similarity:

Based on the vector of 2 words or sentence the similarity score cos(angle) is claculated to check whether they are close.

For high dimentional data one formula is available to find the similarity score.

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(movie_title, n=10):
    # Check if the movie title exists in the dataset
    try:
        if movie_title not in indices:
            raise ValueError(f"Movie title '{movie_title}' not found in the dataset.")

        # Get the index of the movie that matches the title
        idx = indices[movie_title]
        sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten() # Compute the cosine similarity between the movie and all other movies
        similar_idx = sim_scores.argsort()[-n-1:-1][::-1] # Get the indices of the top n similar movies, excluding the movie itself
        return df['title'].iloc[similar_idx].tolist()

    except ValueError as e:
        print(e)
        return []

# Testing:

In [33]:
recommeendations = recommend("Toy Story", n=10) # Getting recommendations for the movie "Toy Story"
print("Recommendations for 'Toy Story':", recommeendations)

Recommendations for 'Toy Story': ['Toy Story 2', 'Toy Story 3', 'Small Fry', 'Superstar Goofy', 'Group Sex', "What's Up, Tiger Lily?", 'For Your Consideration', 'Rebel Without a Cause', 'Condorman', 'Malice']


In [34]:
recommend("Avatar", n=10) # Testing the recommendation function with the movie "Avatar" and requesting 10 similar movies

['Avatar 2',
 'The Inhabited Island',
 'Thor: Ragnarok',
 'Moontrap: Target Earth',
 'The Three Musketeers',
 'A Trip to the Moon',
 'Nightmare City 2035',
 'France société anonyme',
 'Désiré',
 'Stand by Me Doraemon']

In [35]:
recommend("Avengers", n=10) # Testing the recommendation function with the movie "Avengers" and requesting 10 similar movies

Movie title 'Avengers' not found in the dataset.


[]

# Saving The Files

In [ ]:

import pickle

pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb')) # Saving the TF-IDF matrix for later use in the recommendation function

pickle.dump(indices,open('indices.pkl','wb')) # Saving the indices Series for later use in the recommendation function

df.to_pickle('df.pkl') # Saving the processed DataFrame to a pickle file for future use

pickle.dump(tfidf,open('tfidf.pkl','wb')) # Saving the TF-IDF vectorizer for later use in the recommendation function